# 02 — Analyse en composantes principales (ACP)

Le notebook agrège les métriques d'**un seul corpus**, prépare les variables
numériques, impute les valeurs manquantes, standardise les variables puis réalise
une ACP. Les résultats sont écrits dans `analysis/<corpus>/pca/`.


## 1. Configuration


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from metric_registry import get_dataset
from pipeline_utils import detect_project_dir, aggregate_metrics, prepare_numeric_features

PROJECT_DIR = detect_project_dir()
DATASET = "Mirabelle"  # "Mirabelle", "Nowledgeable" ou "Progsnap2"
JOIN_MODE = "outer"
IMPUTATION = "median"  # "median", "mean" ou "drop_rows"
MIN_NON_NULL_FEATURES = 2
DROP_FEATURES = []
RANDOM_STATE = 42  # conservé pour homogénéité ; PCA sklearn est déterministe ici

SPEC = get_dataset(DATASET)
CSV_DIR = SPEC.csv_dir(PROJECT_DIR)
ANALYSIS_DIR = SPEC.analysis_dir(PROJECT_DIR) / "pca"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print("Corpus  :", DATASET)
print("CSV     :", CSV_DIR)
print("Analyse :", ANALYSIS_DIR)


## 2. Agrégation et nettoyage


In [ ]:
features, inventory_df = aggregate_metrics(CSV_DIR, join_mode=JOIN_MODE)
display(inventory_df)

X_raw, usable_cols, removed_df = prepare_numeric_features(
    features,
    drop_features=DROP_FEATURES,
    min_non_null_features=MIN_NON_NULL_FEATURES,
)
if len(usable_cols) < 2:
    raise ValueError("Il faut au moins deux variables numériques non constantes pour une ACP.")
if not removed_df.empty:
    display(removed_df)

missing = X_raw[usable_cols].isna().mean().mul(100).sort_values(ascending=False)
display(missing.rename("pct_manquant").to_frame())


## 3. Imputation et standardisation


In [ ]:
subject_ids = X_raw["SubjectID"].astype(str).reset_index(drop=True)
X_values = X_raw[usable_cols].copy()

if IMPUTATION == "drop_rows":
    keep = X_values.notna().all(axis=1)
    X_values = X_values.loc[keep].reset_index(drop=True)
    subject_ids = subject_ids.loc[keep].reset_index(drop=True)
elif IMPUTATION in {"median", "mean"}:
    imputer = SimpleImputer(strategy=IMPUTATION)
    X_values = pd.DataFrame(imputer.fit_transform(X_values), columns=usable_cols)
else:
    raise ValueError("IMPUTATION doit valoir 'median', 'mean' ou 'drop_rows'.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

features_for_pca = pd.concat([subject_ids.rename("SubjectID"), X_values], axis=1)
features_for_pca.to_csv(ANALYSIS_DIR / "features_for_pca.csv", index=False)
print(f"{len(subject_ids)} étudiant(s), {len(usable_cols)} variable(s).")


## 4. ACP et fichiers de résultats


In [ ]:
pca = PCA()
scores = pca.fit_transform(X_scaled)
component_names = [f"PC{i+1}" for i in range(scores.shape[1])]

explained_df = pd.DataFrame({
    "component": component_names,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "explained_variance_pct": pca.explained_variance_ratio_ * 100,
    "cumulative_variance_pct": np.cumsum(pca.explained_variance_ratio_) * 100,
})

scores_df = pd.DataFrame(scores, columns=component_names)
scores_df.insert(0, "SubjectID", subject_ids.values)

# Les loadings ci-dessous sont les corrélations variable-composante pour des
# données standardisées : composantes * sqrt(valeurs propres).
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loadings_df = pd.DataFrame(loadings, columns=component_names)
loadings_df.insert(0, "variable", usable_cols)

explained_df.to_csv(ANALYSIS_DIR / "explained_variance.csv", index=False)
scores_df.to_csv(ANALYSIS_DIR / "pca_scores.csv", index=False)
loadings_df.to_csv(ANALYSIS_DIR / "pca_loadings.csv", index=False)

display(explained_df.head(10))


## 5. Variance expliquée


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(explained_df["component"], explained_df["explained_variance_pct"], marker="o")
plt.xlabel("Composante")
plt.ylabel("Variance expliquée (%)")
plt.title(f"{DATASET} — variance expliquée par composante")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(explained_df["component"], explained_df["cumulative_variance_pct"], marker="o")
plt.axhline(80, linewidth=1)
plt.xlabel("Composante")
plt.ylabel("Variance cumulée (%)")
plt.title(f"{DATASET} — variance expliquée cumulée")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Projection et contributions principales


In [ ]:
if len(component_names) >= 2:
    plt.figure(figsize=(8, 6))
    plt.scatter(scores_df["PC1"], scores_df["PC2"], alpha=0.75)
    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.xlabel(f"PC1 ({explained_df.loc[0, 'explained_variance_pct']:.1f} %)")
    plt.ylabel(f"PC2 ({explained_df.loc[1, 'explained_variance_pct']:.1f} %)")
    plt.title(f"{DATASET} — projection des étudiants")
    plt.tight_layout()
    plt.show()

for pc in component_names[:min(5, len(component_names))]:
    tmp = loadings_df[["variable", pc]].copy()
    tmp["abs_loading"] = tmp[pc].abs()
    print()
    print(f"Variables les plus liées à {pc}")
    display(tmp.sort_values("abs_loading", ascending=False).head(10).drop(columns="abs_loading"))


## 7. Matrice de corrélation


In [ ]:
corr = features_for_pca[usable_cols].corr()
plt.figure(figsize=(max(8, 0.5 * len(usable_cols)), max(6, 0.5 * len(usable_cols))))
im = plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(usable_cols)), usable_cols, rotation=90)
plt.yticks(range(len(usable_cols)), usable_cols)
plt.title(f"{DATASET} — corrélations entre métriques")
plt.tight_layout()
plt.show()
